In [ ]:
!pip install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-aws \
    faiss-cpu \
    pypdf \
    boto3

In [ ]:
import boto3

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_aws import BedrockEmbeddings
from langchain_aws import ChatBedrock

from langchain.chains import RetrievalQA

In [ ]:
##Configure AWS Credentials
import boto3
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1",
    aws_access_key_id="YOUR_ACCESS_KEY",
    aws_secret_access_key="YOUR_SECRET_KEY"
)

In [ ]:
## Cell 4: Upload PDF

In [ ]:
## Cell 5: Load PDF
loader = PyPDFLoader("Policy.pdf")
documents = loader.load()


In [ ]:
## Cell 6: Split Documents
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50 )
docs = splitter.split_documents(documents)

In [ ]:
## Cell 8: Create FAISS Index
vectorstore = FASISS.from_documents(docs, BedrockEmbeddings(model_id="amazon.titan-embedding-text-v1"))

In [ ]:
## Cell 9: Create Claude LLM
llm = ChatBedrock(
    model_id="anthropic.claude-v2", 
    temperature=0, 
    max_tokens=200
    )

In [ ]:
## Cell 10: Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
##Cell 11: Create RetrievalQA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
##Cell 12: Ask Questions
question = "What is the policy coverage for natural disasters?"
result = qa_chain({"query": question})
print("Answer:", result["result"])

In [ ]:
for doc in result  ["source_documents"]:
    print("="*60)
    print(doc.metadata)
    print(doc.page_content[:500])